# Wave Data Downloader — Marconi Beach, Cape Cod

**Author:** Chelsea Volpano  
**Email:** cvolpano@contractor.usgs.gov | cvolpano@gmail.com  
**Date:** 2026-06-24  
**Created using Claude Sonnet 4.6**

---

Downloads wave data (Oct 2024 – Mar 2025) from:
- **WIS Hindcast** Station 63064 via the WIS Portal API (Oct–Dec 2024)
- **NDBC Buoy 44020** (Nantucket Sound, nearshore — full period)
- **NDBC Buoy 44018** (SE Cape Cod, open Atlantic — 2024 only)

In [4]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
from pathlib import Path

In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────
OUT_DIR = Path("wave_data")
OUT_DIR.mkdir(exist_ok=True)

START = "2006-01-01"
END   = "2026-01-01"

## 1. WIS Hindcast — Station 63064

Uses the WIS Portal API at `wisportal.erdc.dren.mil`, passing the THREDDS OPeNDAP
dataset URL as the `url` parameter. Returns CSV. Data available through end of 2024.

In [6]:
WIS_STATION    = "63064"
WIS_PORTAL_API = "https://wisportal.erdc.dren.mil/data/api/station/data"
WIS_THREDDS    = "https://chldata.erdc.dren.mil/thredds/dodsC/wis/Atlantic"
WIS_VARIABLES  = "waveHs,waveTp,waveTm,waveMeanDirection"

def fetch_wis(station: str, start: str, end: str) -> pd.DataFrame | None:
    dataset_url = f"{WIS_THREDDS}/ST{station}/ST{station}.nc4"
    params = {
        "url":       dataset_url,
        "time":      f"{start}T00:00:00/{end}T23:59:59",
        "format":    "csv",
        "variables": WIS_VARIABLES,
    }
    # Show the full composed URL for verification
    req = requests.Request("GET", WIS_PORTAL_API, params=params).prepare()
    print(f"  Full request URL:\n  {req.url}\n")
    try:
        r = requests.get(WIS_PORTAL_API, params=params, timeout=120)
        r.raise_for_status()
    except Exception as e:
        print(f"  ⚠ Request failed: {e}")
        return None

    lines = [l for l in r.text.splitlines() if not l.strip().startswith("#")]
    df = pd.read_csv(StringIO("\n".join(lines)))
    print(f"  Columns: {list(df.columns)}")

    for dt_col in ["time", "datetime", "date", "TIME", "DATE"]:
        if dt_col in df.columns:
            df["datetime"] = pd.to_datetime(df[dt_col])
            if dt_col != "datetime":
                df = df.drop(columns=[dt_col])
            break
    else:
        print(f"  ⚠ Could not find datetime column. Columns: {list(df.columns)}")
        return df

    df = df.set_index("datetime")
    print(f"  ✓ Retrieved {len(df)} rows")
    return df

print("── WIS Hindcast ──")
wis_clip = fetch_wis(WIS_STATION, START, "2024-12-31")

if wis_clip is not None:
    wis_out = OUT_DIR / f"WIS_ST{WIS_STATION}_{START[:7]}_to_{END[:7]}.csv"
    wis_clip.to_csv(wis_out)
    print(f"  ✓ Saved {len(wis_clip)} rows → {wis_out}")
else:
    print("  ⚠ No WIS data retrieved.")

── WIS Hindcast ──
  Full request URL:
  https://wisportal.erdc.dren.mil/data/api/station/data?url=https%3A%2F%2Fchldata.erdc.dren.mil%2Fthredds%2FdodsC%2Fwis%2FAtlantic%2FST63064%2FST63064.nc4&time=2006-01-01T00%3A00%3A00%2F2024-12-31T23%3A59%3A59&format=csv&variables=waveHs%2CwaveTp%2CwaveTm%2CwaveMeanDirection

  Columns: ['time', 'lat', 'lon', 'waveTp', 'waveHs', 'waveTm', 'waveMeanDirection']
  ✓ Retrieved 166549 rows
  ✓ Saved 166549 rows → wave_data/WIS_ST63064_2006-01_to_2026-01.csv


In [7]:
wis_clip.head()

,lat,lon,waveTp,waveHs,waveTm,waveMeanDirection
datetime,,,,,,
2006-01-01 00:00:00,41.916672,-69.75,9.507812,1.031250,7.859375,97.992188
2006-01-01 01:00:00,41.916672,-69.75,9.523438,1.015625,7.804688,98.945312
2006-01-01 02:00:00,41.916672,-69.75,9.515625,1.015625,7.609375,98.023438
2006-01-01 03:00:00,41.916672,-69.75,9.500000,1.023438,7.351562,95.570312
2006-01-01 04:00:00,41.916672,-69.75,9.476562,1.054688,6.945312,90.859375


## 2. NDBC Buoy Data

Fetches standard meteorological (stdmet) data from NOAA NDBC historical archive.  
Key wave columns: **WVHT** (Hs, m), **DPD** (dominant period, s), **APD** (avg period, s), **MWD** (mean direction, °).

| Buoy | Location | Notes |
|------|----------|-------|
| 44020 | Nantucket Sound | Primary — full period available |
| 44018 | SE Cape Cod (~60 nm offshore) | 2024 only |

In [12]:
import gzip
import shutil
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

NDBC_BASE  = "https://www.ndbc.noaa.gov/data/historical/stdmet"
NDBC_BUOYS = {
    "44020": "Nantucket Sound (primary)",
    "44013": "16 NM East of Boston, MA",
    "44008": "54 NM Southeast of Nantucket",
}

def get_available_years(buoy: str) -> list:
    """Scrape NDBC historical stdmet directory for available years for a buoy."""
    r    = requests.get(f"{NDBC_BASE}/", timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")
    years = []
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if href.startswith(buoy) and href.endswith(".txt.gz"):
            try:
                years.append(int(href.replace(f"{buoy}h", "").replace(".txt.gz", "")))
            except ValueError:
                pass
    return sorted(years)

def fetch_ndbc(buoy: str, year: int) -> pd.DataFrame | None:
    url      = f"{NDBC_BASE}/{buoy}h{year}.txt.gz"
    gz_path  = OUT_DIR / f"NDBC_{buoy}_{year}.txt.gz"
    txt_path = OUT_DIR / f"NDBC_{buoy}_{year}.txt"
    print(f"  Fetching NDBC {buoy} {year}: {url}")

    # ── Download and unzip ────────────────────────────────────────────────────
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        gz_path.write_bytes(r.content)
        with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        gz_path.unlink()
    except Exception as e:
        print(f"  ⚠ Could not download {buoy} {year}: {e}")
        return None

    # ── Parse ─────────────────────────────────────────────────────────────────
    try:
        with open(txt_path, "r") as f:
            lines = f.readlines()

        # Line 0: column names (strip leading #)
        # Line 1: units row — always skip
        # Line 2+: data
        col_names  = lines[0].strip().lstrip("#").split()
        data_lines = lines[2:]

        df = pd.read_csv(
            StringIO("".join(data_lines)),
            sep=r"\s+", header=None, names=col_names,
            na_values=["99", "999", "9999", "99.0", "999.0"],
            low_memory=False, dtype=str
        )
        df.columns = df.columns.str.strip().str.lstrip("#")
        txt_path.unlink()

    except Exception as e:
        print(f"  ⚠ Could not read {buoy} {year}: {e}")
        txt_path.unlink(missing_ok=True)
        return None

    # ── Build datetime ────────────────────────────────────────────────────────
    try:
        # Identify year column and whether minutes exist
        year_col = "YYYY" if "YYYY" in df.columns else "YY"
        has_min  = "mm" in df.columns
        time_cols = [year_col, "MM", "DD", "hh"] + (["mm"] if has_min else [])

        # All columns are already strings (dtype=str), so len() is safe
        year_len = len(df[year_col].iloc[0].strip())
        if has_min:
            fmt = "%Y-%m-%d-%H-%M" if year_len == 4 else "%y-%m-%d-%H-%M"
        else:
            fmt = "%Y-%m-%d-%H"    if year_len == 4 else "%y-%m-%d-%H"

        time_str = df[time_cols].apply(lambda r: "-".join(v.strip() for v in r), axis=1)
        df["datetime"] = pd.to_datetime(time_str, format=fmt, errors="coerce")

    except Exception as e:
        print(f"  ⚠ Could not parse datetime for {buoy} {year}: {e}")
        return None

    # ── Finalise ──────────────────────────────────────────────────────────────
    df = df.set_index("datetime")
    df = df[~df.index.duplicated(keep="first")]
    df = df[~df.index.isna()]

    wave_cols = [c for c in ["WVHT", "DPD", "APD", "MWD"] if c in df.columns]
    if not wave_cols:
        print(f"  ⚠ No wave columns found for {buoy} {year}")
        return None

    print(f"  ✓ {len(df)} rows | {df.index.min()} → {df.index.max()} | cols: {wave_cols}")
    return df[wave_cols]


# ── Main download loop ────────────────────────────────────────────────────────
print("── NDBC Buoys ──")
ndbc_dfs = {}
for buoy, desc in NDBC_BUOYS.items():
    print(f"\n  Buoy {buoy} — {desc}")
    years  = get_available_years(buoy)
    print(f"  Available years: {years}")
    frames = [fetch_ndbc(buoy, y) for y in years]
    frames = [f for f in frames if f is not None]
    if not frames:
        print(f"  ⚠ No data retrieved for buoy {buoy}.")
        continue
    combined = pd.concat(frames).sort_index()
    combined = combined[~combined.index.duplicated(keep="first")]
    clipped  = combined.loc[START:END].copy()
    ndbc_dfs[buoy] = clipped
    out_path = OUT_DIR / f"NDBC_{buoy}_{START[:7]}_to_{END[:7]}.csv"
    clipped.to_csv(out_path)
    print(f"  ✓ Saved {len(clipped)} rows → {out_path}")

── NDBC Buoys ──

  Buoy 44020 — Nantucket Sound (primary)
  Available years: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
  Fetching NDBC 44020 2009: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2009.txt.gz
  ✓ 6022 rows | 2009-03-10 13:50:00 → 2009-12-31 22:50:00 | cols: ['WVHT', 'DPD', 'APD', 'MWD']
  Fetching NDBC 44020 2010: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2010.txt.gz
  ✓ 8654 rows | 2009-12-31 23:50:00 → 2010-12-31 22:50:00 | cols: ['WVHT', 'DPD', 'APD', 'MWD']
  Fetching NDBC 44020 2011: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2011.txt.gz
  ✓ 8708 rows | 2010-12-31 23:50:00 → 2011-12-31 22:50:00 | cols: ['WVHT', 'DPD', 'APD', 'MWD']
  Fetching NDBC 44020 2012: https://www.ndbc.noaa.gov/data/historical/stdmet/44020h2012.txt.gz
  ✓ 8604 rows | 2011-12-31 23:50:00 → 2012-12-31 22:50:00 | cols: ['WVHT', 'DPD', 'APD', 'MWD']
  Fetching NDBC 44020 2013: https://www.ndbc.noaa.gov/d

In [ ]:
ndbc_dfs["44020"].head()

,WVHT,DPD,APD,MWD
datetime,,,,
2024-01-01 00:00:00,99.00,99.00,99.00,NaN
2024-01-01 00:10:00,0.34,2.86,2.92,274
2024-01-01 00:20:00,99.00,99.00,99.00,NaN
2024-01-01 00:30:00,99.00,99.00,99.00,NaN
2024-01-01 00:40:00,0.33,2.94,2.96,261


## 3. Summary Statistics

In [ ]:
if wis_clip is not None:
    print("── WIS ST63064 ──")
    print(wis_clip.describe().round(2))

print("\n── NDBC 44020 ──")
print(ndbc_dfs["44020"].describe().round(2))
print("\n── NDBC 44013 ──")
print(ndbc_dfs["44013"].describe().round(2))
print("\n── NDBC 44008 ──")
print(ndbc_dfs["44008"].describe().round(2))

── WIS ST63064 ──
             lat        lon     waveTp     waveHs     waveTm  \
count  166549.00  166549.00  166549.00  166549.00  166549.00   
mean       41.92     -69.75       7.39       1.19       6.08   
std         0.00       0.00       2.08       0.75       1.36   
min        41.92     -69.75       0.00       0.11       2.52   
25%        41.92     -69.75       5.91       0.69       5.07   
50%        41.92     -69.75       7.37       0.99       5.93   
75%        41.92     -69.75       8.80       1.48       6.90   
max        41.92     -69.75      20.40       7.95      13.99   

       waveMeanDirection  
count          166549.00  
mean              142.42  
std                83.17  
min                 0.00  
25%                90.51  
50%               131.88  
75%               163.46  
max               359.99  

── NDBC 44020 ──
          WVHT     DPD     APD    MWD
count   105252  105252  105252  26984
unique     223      41     293    359
top      99.00   99.00   99.00